# Video Daily Stats — Silver Transformation

- Bronze → Silver
- Inspect Bronze video daily stats
- Exclude Auto Loader rescue column
- Standardize data types
- Write Silver Delta table
- Validate Silver data

## 1. Inspect Bronze data

In [0]:
%sql
-- Preview Bronze data
SELECT *
FROM youtube_content_intelligence.bronze.brz_video_daily_stats
LIMIT 5;

In [0]:
%sql
-- Inspect Bronze schema
DESCRIBE TABLE youtube_content_intelligence.bronze.brz_video_daily_stats;

## 2. Transform Bronze data

In [0]:
from pyspark.sql.functions import col, to_date

# Load Bronze table
df_bronze = spark.table("youtube_content_intelligence.bronze.brz_video_daily_stats")

# Exclude Auto Loader rescue column and standardize data types
df_silver = df_bronze.select(
    to_date("collection_date").alias("collection_date"),
    col("comment_count").cast("long").alias("comment_count"),
    col("like_count").cast("long").alias("like_count"),
    col("video_id"),
    col("view_count").cast("long").alias("view_count"),
    col("ingested_at")
)

In [0]:
display(df_silver.limit(5))

## 3. Write to Silver

In [0]:
df_silver.write.format("delta").mode("overwrite").saveAsTable(
    "youtube_content_intelligence.silver.slv_video_daily_stats"
)

## 4. Validate Silver table

In [0]:
%sql
-- Preview Silver data
SELECT *
FROM youtube_content_intelligence.silver.slv_video_daily_stats
LIMIT 5;

In [0]:
%sql
-- Check row count
SELECT COUNT(*) AS row_count
FROM youtube_content_intelligence.silver.slv_video_daily_stats;

In [0]:
%sql
-- Check duplicate daily records
SELECT video_id, collection_date, COUNT(*) AS count
FROM youtube_content_intelligence.silver.slv_video_daily_stats
GROUP BY video_id, collection_date
HAVING COUNT(*) > 1;